<div style="border-left:6px solid #ae0000; padding:6px 20px; margin-bottom:4px;">
<h1 style="margin:0; color:#0d2741;">Medidas de Dispersión y Forma</h1>
<p style="margin:2px 0 0 0; color:#0d2741; font-size:1.15em;">Variabilidad, Asimetría y Curtosis</p>
<p style="margin:6px 0 0 0; color:#444; font-size:1.05em;"><em>Estadística Computacional para la Toma de Decisiones</em></p>
</div>

**Magíster en Ciencia de Datos e Inteligencia Artificial** · Universidad Andrés Bello  
Maidana, J.P. (2026)

---

> **Cómo usar este notebook.** Ejecuta las celdas en orden (de arriba hacia abajo). Comienza por la celda **«Preparación del entorno»**, que carga las librerías y fija el estilo de los gráficos. Las celdas de código reproducen el material del apunte y son auto-contenidas dentro de cada sección.

### Preparación del entorno

In [ ]:
# Librerías base usadas en todo el notebook
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Estilo de gráficos coherente con el apunte
plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 11,
})

UNAB_NAVY = "#0d2741"
UNAB_RED  = "#ae0000"

print("Entorno listo")
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", stats.__name__)

## 1. Introducción: la insuficiencia de la media

Considérese la siguiente situación: dos equipos de ventas presentan exactamente el mismo desempeño promedio.

| Equipo | \multicolumn{5}{Ventas mensuales (\$K)} | | | | | Media |
|---|---|---|---|---|---|---|
| **A** | 48 | 49 | 50 | 51 | 52 | **50** |
| **B** | 10 | 30 | 50 | 70 | 90 | **50** |

La media es idéntica: \$50K. Sin embargo, los equipos son cualitativamente distintos. El **Equipo A** es altamente predecible: sus ventas varían entre \$48K y \$52K, lo que permite planificar presupuesto con precisión. El **Equipo B** oscila entre \$10K y \$90K: la media de \$50K no es representativa de ningún mes real y hace que cualquier proyección sea poco confiable.

Esta distinción ilustra el concepto central de este apunte: la **variabilidad**. La media informa sobre el *centro* de los datos, pero la variabilidad informa sobre *qué tan confiable* es ese centro. Reportar únicamente la media sin contexto de dispersión proporciona una imagen incompleta y potencialmente engañosa.

*Visualización del ejemplo de los dos equipos:*

In [ ]:
A = np.array([48, 49, 50, 51, 52])
B = np.array([10, 30, 50, 70, 90])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, d, nom, col in zip(axes, [A, B], ["Equipo A", "Equipo B"], ["#2e8b57", UNAB_RED]):
    ax.bar(range(1, 6), d, color=col, alpha=0.85, edgecolor="white")
    ax.axhline(50, ls="--", color=UNAB_NAVY, lw=1.5, label="Media = 50")
    ax.set_title(f"{nom}  (σ = {d.std(ddof=1):.1f})", color=UNAB_NAVY)
    ax.set_xlabel("Mes"); ax.set_xticks(range(1, 6)); ax.legend(fontsize=8)
axes[0].set_ylabel("Ventas (miles USD)")
fig.suptitle("Misma media, dispersión muy distinta", fontsize=13,
             fontweight="bold", color=UNAB_RED)
plt.tight_layout(); plt.show()

### 1.1 Relevancia en distintos contextos

La variabilidad tiene implicaciones concretas en múltiples dominios:

- **Finanzas.** Dos fondos con retorno promedio anual del 8 %. Si el primero tiene desviación estándar de 2 % y el segundo de 15 %, el perfil de riesgo es radicalmente diferente. La variabilidad *cuantifica el riesgo*.
- **Control de calidad.** Un proceso con media perfecta de 10,0 mm pero desviación estándar de 0,15 mm producirá entre un 15 % y 20 % de piezas fuera de especificación (tolerancia ±0,1 mm). La media no revela este problema.
- **Machine Learning.** Variables con rangos muy distintos (por ejemplo, edad: 20–80 vs. salario: 20.000–200.000) pueden sesgar modelos basados en distancias o gradientes. La estandarización mediante media y desviación estándar resuelve este problema.
- **Detección de anomalías.** Un tiempo de respuesta de 300 ms en un sistema con media de 200 ms puede ser normal o crítico dependiendo de la desviación estándar. Sin medir dispersión, no es posible establecer umbrales de alerta confiables.

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Objetivos de aprendizaje</p>

<ol style="margin:4px 0 0 0;">
<li>Calcular e interpretar medidas de dispersión: rango, IQR, varianza y desviación estándar.</li>
<li>Comprender la corrección de Bessel ($n-1$) y su justificación estadística.</li>
<li>Medir asimetría y curtosis para caracterizar la forma de una distribución.</li>
<li>Implementar estos cálculos en Python con NumPy, SciPy y pandas.</li>
<li>Aplicar estas medidas en contextos reales de análisis de datos.</li>
</ol>
</div>

## 2. El mapa completo de los estadísticos descriptivos

Los estadísticos descriptivos se organizan en tres categorías complementarias.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 12); ax.set_ylim(1.6, 7.8); ax.axis("off")

def _box(x, y, w, h, text, fc, tc="white", fs=10):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h,
                 boxstyle="round,pad=0.02,rounding_size=0.12",
                 fc=fc, ec="white", lw=1.5, zorder=2))
    ax.text(x, y, text, ha="center", va="center", color=tc,
            fontsize=fs, fontweight="bold", zorder=3)

def _line(x1, y1, x2, y2):
    ax.plot([x1, x2], [y1, y2], color="#888", lw=1.4, zorder=1)

_box(6, 7, 3.4, 0.9, "Estadísticos\nDescriptivos", "#7b4fa3")
for t, (x, y, c) in {"Tendencia\nCentral": (2.5, 4.6, "#2e6fb0"),
                     "Dispersión": (6, 4.6, "#2e8b57"),
                     "Forma": (9.5, 4.6, "#d98a00")}.items():
    _line(6, 6.55, x, 5.05); _box(x, y, 2.6, 0.85, t, c)

_box(2.5, 2.4, 2.9, 0.8, "Media, Mediana, Moda", "#cfe0f0", "#13314f", 8.5)
_line(2.5, 4.18, 2.5, 2.8)
_box(4.7, 2.4, 2.4, 0.8, "Rango, IQR", "#cbe6d4", "#14502c", 8.5)
_line(5.6, 4.18, 4.7, 2.8)
_box(7.3, 2.4, 2.4, 0.8, "Varianza, σ²", "#cbe6d4", "#14502c", 8.5)
_line(6.4, 4.18, 7.3, 2.8)
_box(8.8, 2.4, 2.1, 0.8, "Asimetría", "#f6e2c0", "#7a4d00", 8.5)
_line(9.2, 4.18, 8.8, 2.8)
_box(10.9, 2.4, 2.0, 0.8, "Curtosis", "#f6e2c0", "#7a4d00", 8.5)
_line(9.8, 4.18, 10.9, 2.8)

ax.set_title("Mapa de los estadísticos descriptivos", fontsize=13,
             color=UNAB_NAVY, pad=10)
plt.tight_layout(); plt.show()

Cada categoría responde una pregunta diferente:

- **Tendencia central**: ¿dónde se localiza el centro de los datos?
- **Dispersión**: ¿qué tan esparcidos están los datos respecto al centro?
- **Forma**: ¿la distribución es simétrica? ¿Presenta colas pesadas?

Las tres categorías son necesarias para una descripción completa. Utilizar solo una o dos produce una imagen parcial de la distribución.

## 3. Medidas de dispersión

### 3.1 Rango y rango intercuartílico

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Rango e IQR</p>

El **rango** es la diferencia entre el valor máximo y el mínimo:

$$R = x_{\max} - x_{\min}$$

El **rango intercuartílico (IQR)** es una medida más robusta:

$$IQR = Q_3 - Q_1$$

El IQR mide el rango del 50 % central de los datos, ignorando el 25 % inferior y el 25 % superior.
</div>

**Utilidad del rango.** Proporciona una idea inmediata de la amplitud total de los datos y es fácil de comunicar a audiencias no técnicas. Sin embargo, tiene una limitación fundamental: un único valor atípico puede distorsionarlo completamente.

Ejemplo ilustrativo: el conjunto $\{1, 2, 3, 4, 100\}$ tiene rango de 99, cuando el 80 % de los datos se concentra entre 1 y 4.

**Ventaja del IQR.** Al basarse en cuartiles, el IQR no se ve afectado por valores extremos. Para el mismo ejemplo, $Q_1 = 2$, $Q_3 = 4$, $IQR = 2$, lo que refleja con mayor fidelidad la dispersión del grueso de los datos.

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Aplicaciones del rango e IQR</p>

<b>Finanzas.</b> El diferencial <i>bid-ask</i> (precio de compra menos precio de venta) es un rango que indica la liquidez del mercado. Un diferencial pequeño indica alta liquidez.<br><br>
<b>Meteorología.</b> La amplitud térmica diaria (máxima menos mínima) puede superar los 30 °C en zonas desérticas, frente a 10 °C en zonas costeras.<br><br>
<b>Control de calidad.</b> Si la tolerancia de una pieza es ±0,5 mm, el rango de producción aceptable es 1 mm. Un rango mayor indica problemas de proceso.<br><br>
<b>Detección de outliers con IQR (regla de Tukey):</b>
<ul>
<li>Outlier leve: $x < Q_1 - 1{,}5 \times IQR$ &nbsp; o &nbsp; $x > Q_3 + 1{,}5 \times IQR$</li>
<li>Outlier extremo: $x < Q_1 - 3 \times IQR$ &nbsp; o &nbsp; $x > Q_3 + 3 \times IQR$</li>
</ul>
El factor 1,5 fue establecido por John Tukey como convención práctica que funciona bien en la mayoría de distribuciones reales.
</div>

### 3.2 Varianza

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Varianza</p>

La **varianza** mide la dispersión promedio cuadrática respecto a la media.

**Varianza poblacional** (cuando se dispone de todos los datos):

$$\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i - \mu)^2$$

**Varianza muestral** (cuando se trabaja con una muestra):

$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$
</div>

**Justificación del cuadrado.** Si se suman las desviaciones simples respecto a la media, el resultado siempre es cero, ya que las desviaciones positivas y negativas se cancelan exactamente. Por ejemplo, con los datos $\{8, 10, 12\}$ y media $= 10$: las desviaciones son $\{-2, 0, +2\}$, que suman cero. Elevar al cuadrado garantiza que todas las contribuciones sean positivas y que desviaciones mayores tengan mayor peso.

**Propiedades algebraicas de la varianza:**
1. No negatividad: $\sigma^2 \geq 0$
2. Fórmula equivalente: $\sigma^2 = \overline{x^2} - \bar{x}^2$
3. Escalamiento: $\operatorname{Var}(bX) = b^2 \operatorname{Var}(X)$
4. Aditividad para variables independientes: $\operatorname{Var}(X+Y) = \operatorname{Var}(X) + \operatorname{Var}(Y)$

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — La corrección de Bessel: por qué se divide por $n-1$</p>

Esta es una de las preguntas más frecuentes al estudiar estadística descriptiva, y tiene una respuesta bien fundamentada.<br><br>
<b>Explicación técnica.</b> Al dividir por $n$ en lugar de $n-1$, se obtiene un estimador sistemáticamente subestimado de la varianza poblacional $\sigma^2$. Es decir, $E[s_n^2] < \sigma^2$. El denominador $n-1$ corrige este sesgo, logrando que $E[s^2] = \sigma^2$.<br><br>
<b>Explicación intuitiva.</b> Al calcular $\bar{x}$ a partir de los mismos datos, ya se ha "consumido" un grado de libertad. Los datos están restringidos a tener esa media específica, por lo que solo quedan $n-1$ desviaciones independientes.<br><br>
<b>Impacto práctico según el tamaño muestral:</b>
<ul>
<li>$n = 5$: diferencia entre $n$ y $n-1$ de 25 % — relevante</li>
<li>$n = 30$: diferencia de 3,4 % — menor pero no despreciable</li>
<li>$n = 1000$: diferencia de 0,1 % — prácticamente insignificante</li>
</ul>
En muestras pequeñas, usar $n$ en lugar de $n-1$ introduce un error apreciable. La buena práctica es emplear siempre $n-1$ al trabajar con muestras.<br><br>
<b>Diferencia en Python — aspecto crítico:</b>
<ul>
<li><b>NumPy</b>: <code>ddof=0</code> por defecto (divide por $n$, asume población completa).</li>
<li><b>Pandas</b>: <code>ddof=1</code> por defecto (divide por $n-1$, asume muestra).</li>
</ul>
Esta diferencia de comportamiento entre librerías es una fuente frecuente de errores sutiles. Se recomienda especificar <code>ddof</code> explícitamente en todo código de producción.
</div>

### 3.3 Desviación estándar

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Desviación estándar</p>

La **desviación estándar** es la raíz cuadrada de la varianza:

$$s = \sqrt{s^2} = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$
</div>

La razón de preferir la desviación estándar sobre la varianza es **dimensional**: si los datos están en metros, la varianza está en metros cuadrados, unidad que carece de interpretación directa. La desviación estándar "deshace" el cuadrado y recupera las unidades originales.

**Ventajas de la desviación estándar:**
- Tiene las mismas unidades que los datos, lo que facilita la interpretación.
- Se puede leer como "en promedio, los datos se alejan $s$ unidades de la media".
- Permite aplicar la regla empírica para distribuciones aproximadamente normales.

**Regla empírica 68–95–99,7 (distribuciones aproximadamente normales):**
- Aproximadamente el 68 % de los datos cae en $[\bar{x} - s,\ \bar{x} + s]$.
- Aproximadamente el 95 % cae en $[\bar{x} - 2s,\ \bar{x} + 2s]$.
- Aproximadamente el 99,7 % cae en $[\bar{x} - 3s,\ \bar{x} + 3s]$.

Ejemplo de aplicación: tiempos de respuesta de una API con media de 200 ms y desviación estándar de 50 ms. El 95 % de las solicitudes se espera entre 100 ms y 300 ms. Una observación de 400 ms se encuentra a cuatro desviaciones estándar de la media y merece investigación.

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Aplicaciones de la desviación estándar</p>

<b>Finanzas.</b> En mercados financieros, la volatilidad de un activo se define precisamente como la desviación estándar de sus retornos. El índice VIX mide la volatilidad implícita del S&P 500.<br><br>
<b>Six Sigma.</b> Este programa de mejora de calidad define sus límites de control como $\mu \pm 6\sigma$. Alcanzar "seis sigma" implica un proceso tan estable que prácticamente ninguna pieza queda fuera de especificación.<br><br>
<b>Machine Learning.</b> La estandarización de variables mediante $z = (x - \bar{x})/s$ es un paso de preprocesamiento fundamental antes de entrenar modelos sensibles a la escala (regresión logística, SVM, redes neuronales).<br><br>
<b>A/B Testing.</b> El cálculo de significancia estadística requiere la desviación estándar de cada grupo para construir intervalos de confianza y determinar si la diferencia observada entre variantes es atribuible al azar.
</div>

### 3.4 Coeficiente de variación

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Coeficiente de variación (CV)</p>

El **coeficiente de variación (CV)** expresa la dispersión en términos relativos a la media:

$$CV = \frac{s}{|\bar{x}|} \times 100\%$$
</div>

El CV permite comparar la variabilidad de conjuntos de datos medidos en escalas completamente diferentes. Ejemplo:

- Acción A: precio medio \$10, desviación \$2 $\Rightarrow$ CV = 20 %
- Acción B: precio medio \$100, desviación \$15 $\Rightarrow$ CV = 15 %

En términos absolutos, la Acción B fluctúa más (\$15 vs \$2). Sin embargo, en términos relativos, la Acción A es más volátil (20 % vs 15 %). La comparación correcta requiere el CV.

**Interpretación orientativa:**
- CV $< 15\%$: variabilidad baja, proceso consistente.
- CV entre 15 % y 30 %: variabilidad moderada.
- CV $> 30\%$: variabilidad alta, datos muy dispersos.

**Limitaciones del CV:**
- Solo es apropiado para escalas de razón (donde el cero indica ausencia de la magnitud).
- No debe usarse con temperaturas en Celsius, ya que el 0 °C no representa ausencia de temperatura.
- No está definido cuando $\bar{x} = 0$ y es inestable cuando $\bar{x}$ es cercano a cero.

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Aplicación del CV en control de calidad</p>

Considérese una fábrica que produce dos tipos de tornillos:
<ul>
<li>Tornillo pequeño: media = 5 mm, desviación = 0,3 mm $\rightarrow$ CV = 6 %</li>
<li>Tornillo grande: media = 50 mm, desviación = 1,2 mm $\rightarrow$ CV = 2,4 %</li>
</ul>
En términos absolutos, el tornillo grande presenta cuatro veces más variabilidad (1,2 mm vs 0,3 mm). Sin embargo, el CV revela que el proceso del tornillo pequeño es <b>menos consistente</b> en términos relativos (6 % vs 2,4 %). Esta distinción es relevante para priorizar inversiones en mejora de procesos.
</div>

## 4. Tabla resumen de medidas de dispersión

| Medida | Fórmula | Unidades | Robusta | Cuándo usar |
|---|---|---|---|---|
| Rango | $x_{\max} - x_{\min}$ | Originales | No | Idea rápida, comunicar rangos |
| IQR | $Q_3 - Q_1$ | Originales | **Sí** | Datos con outliers, boxplots |
| Varianza | $s^2$ | Cuadradas | No | Teoría estadística, álgebra |
| Desv. estándar | $s$ | Originales | No | Reportar dispersión (estándar) |
| CV | $s/\bar{x} \times 100\%$ | % | No | Comparar escalas distintas |

**Criterios de selección resumidos:**
- **Para reportar**: desviación estándar (comprensible y en las mismas unidades).
- **Para detectar outliers**: IQR (robusto ante valores extremos).
- **Para comparar variables en distintas escalas**: CV (relativiza la dispersión).
- **Para desarrollos teóricos**: varianza (propiedades algebraicas convenientes).

## 5. Medidas de forma

Conocer el centro y la dispersión no agota la información disponible en una distribución. La **forma** aporta información adicional sobre asimetría y frecuencia de eventos extremos.

### 5.1 Asimetría (*skewness*)

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Asimetría</p>

La **asimetría** cuantifica el grado en que una distribución se aleja de la simetría:

$$g_1 = \frac{1}{n}\sum_{i=1}^{n}\left(\frac{x_i - \bar{x}}{s}\right)^3$$

El exponente impar (3) preserva el signo de las desviaciones, permitiendo distinguir la dirección de la asimetría.
</div>

**Interpretación del valor:**
- $|g_1| < 0{,}5$: distribución aproximadamente simétrica.
- $0{,}5 \leq |g_1| < 1$: asimetría moderada.
- $|g_1| \geq 1$: asimetría pronunciada.

**Interpretación del signo:**
- $g_1 > 0$: **asimetría positiva** (cola hacia la derecha). La mayoría de los valores se concentra en valores bajos, con algunos valores muy altos que extienden la cola derecha.
- $g_1 < 0$: **asimetría negativa** (cola hacia la izquierda). La mayoría de los valores se concentra en la parte alta, con algunos valores muy bajos que extienden la cola izquierda.

Una forma intuitiva de entenderlo: la distribución de ingresos tiene asimetría positiva pronunciada, ya que la mayoría de la población percibe ingresos moderados, mientras que una minoría percibe valores muy altos que extienden la cola derecha. La distribución de edad al fallecimiento (excluyendo causas accidentales) tiende a presentar asimetría negativa leve.

*Visualización de las tres formas de asimetría:*

In [ ]:
rng = np.random.default_rng(7)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))

d_pos = rng.exponential(1.0, 5000)            # cola derecha
d_sim = rng.normal(0, 1, 5000)               # simétrica
d_neg = -rng.exponential(1.0, 5000)          # cola izquierda

for ax, d, titulo, col in zip(
        axes, [d_pos, d_sim, d_neg],
        ["Asimetría positiva\n(cola derecha)",
         "Aproximadamente\nsimétrica",
         "Asimetría negativa\n(cola izquierda)"],
        ["#2e6fb0", "#2e8b57", "#d98a00"]):
    ax.hist(d, bins=40, color=col, alpha=0.8)
    ax.set_title(f"{titulo}\n$g_1$ = {stats.skew(d):.2f}", fontsize=10)
    ax.set_yticks([])

plt.tight_layout(); plt.show()

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Distribuciones asimétricas en la práctica</p>

<b>Asimetría positiva (cola derecha):</b>
<ul>
<li>Distribución de ingresos y patrimonios</li>
<li>Precios de viviendas en una ciudad</li>
<li>Tiempos de respuesta de servidores (mayoría rápidos, algunos se demoran)</li>
<li>Tamaño de archivos almacenados (muchos pequeños, algunos muy grandes)</li>
</ul>
<b>Asimetría negativa (cola izquierda):</b>
<ul>
<li>Edad de jubilación (mayoría entre 60–70 años, algunos se jubilan antes)</li>
<li>Puntuaciones en evaluaciones sencillas (mayoría en el rango alto, pocos muy bajos)</li>
<li>Vida útil de productos de alta calidad (mayoría dura mucho, algunos fallan pronto)</li>
</ul>
<b>Distribuciones aproximadamente simétricas:</b>
<ul>
<li>Altura y peso de poblaciones homogéneas</li>
<li>Errores de medición aleatorios</li>
<li>Suma de muchas variables aleatorias independientes (por el TLC)</li>
</ul>
</div>

### 5.2 Curtosis (*kurtosis*): el peso de las colas

<div style="background-color:#e8f1fb; border-left:5px solid #1565c0; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d47a1; font-size:1.05em;">📘&nbsp; Definición — Curtosis</p>

La **curtosis** cuantifica el peso de las colas de una distribución:

$$g_2 = \frac{\dfrac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^4}{s^4} - 3$$

El término $-3$ normaliza el resultado de modo que la distribución normal tenga curtosis igual a cero (**curtosis de exceso**).
</div>

<div style="background-color:#fdecea; border-left:5px solid #c62828; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#b71c1c; font-size:1.05em;">⚠️&nbsp; Advertencia — La curtosis mide el peso de las colas, no la altura del pico</p>

Es frecuente encontrar —incluso en algunos textos introductorios— la afirmación de que la curtosis mide cuán "picuda" o "achatada" es la distribución. Esta descripción es <b>incorrecta</b>. La investigación estadística moderna es clara al respecto: la curtosis es una medida del <b>peso relativo de las colas</b> de la distribución.<br><br>
Curtosis alta ($g_2 > 0$) significa que existen más observaciones extremas de las que se esperarían bajo una distribución normal. Curtosis baja ($g_2 < 0$) indica lo contrario: los datos están más concentrados en torno a la media que en una distribución normal.
</div>

**Clasificación:**
- $g_2 \approx 0$: **mesocúrtica** (similar a la distribución normal).
- $g_2 > 0$: **leptocúrtica** (colas pesadas, mayor frecuencia de eventos extremos).
- $g_2 < 0$: **platicúrtica** (colas ligeras, menor frecuencia de eventos extremos).

**Relevancia práctica de la curtosis:**
- **Finanzas.** Los retornos de activos exhiben típicamente curtosis positiva alta (leptocúrtica). Los eventos extremos —como caídas de mercado severas— ocurren con mayor frecuencia de la que predice un modelo normal. Es el fundamento estadístico del concepto de "cisne negro" de Nassim Taleb.
- **Seguros.** Una curtosis alta en la distribución de siniestros indica que los eventos catastróficos (huracanes, terremotos) son más probables de lo que un modelo normal estimaría.
- **Machine Learning.** Si se asume normalidad en los errores del modelo pero la distribución real es leptocúrtica, los intervalos de predicción estarán subestimados en los extremos.
- **Control de calidad.** Una curtosis baja sugiere un proceso estable con pocos defectos extremos. Una curtosis alta indica inestabilidad y mayor probabilidad de producir piezas muy fuera de especificación.

<div style="background-color:#e9f7ea; border-left:5px solid #2e7d32; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#1b5e20; font-size:1.05em;">✅&nbsp; Ejemplo — Curtosis en retornos financieros</p>

Un análisis de retornos diarios del S&P 500 durante 20 años típicamente arroja resultados como los siguientes:
<ul>
<li>Media: $\approx 0{,}03\%$ diario</li>
<li>Desviación estándar: $\approx 1{,}2\%$</li>
<li>Asimetría: $\approx -0{,}35$ (ligera cola izquierda: las caídas son más abruptas que las subidas)</li>
<li>Curtosis: $\approx 8{,}5$ (muy alta: los eventos extremos son mucho más frecuentes que bajo normalidad)</li>
</ul>
Esta curtosis elevada tiene una implicación directa: los modelos financieros basados en distribución normal <b>subestiman sistemáticamente</b> el riesgo de pérdidas extremas. Bajo distribución normal, un evento "seis sigma" debería ocurrir una vez cada varios miles de años; en la práctica, eventos de esa magnitud se han observado en múltiples ocasiones durante el siglo XX.
</div>

## 6. Implementación en Python

### 6.1 NumPy: cálculo de dispersión

In [ ]:
datos = np.array([23, 45, 67, 89, 34, 56, 78, 90, 12, 45, 67, 89, 23])

print("=" * 60)
print("MEDIDAS DE DISPERSIÓN")
print("=" * 60)

# Rango
rango = np.ptp(datos)
print(f"Rango: {rango}")

# IQR
Q1 = np.percentile(datos, 25)
Q3 = np.percentile(datos, 75)
IQR = Q3 - Q1
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")

# Varianza: especificar ddof siempre
# ddof=0 -> población (divide por n)
# ddof=1 -> muestra (divide por n-1) — uso estándar en análisis
var_poblacion = np.var(datos, ddof=0)
var_muestra   = np.var(datos, ddof=1)
print(f"\nVarianza poblacional (ddof=0): {var_poblacion:.2f}")
print(f"Varianza muestral   (ddof=1): {var_muestra:.2f}")
print(f"Diferencia: {var_muestra - var_poblacion:.2f}")

# Desviación estándar muestral
std_muestra = np.std(datos, ddof=1)
print(f"\nDesviación estándar (muestra): {std_muestra:.2f}")

# Coeficiente de variación
media = np.mean(datos)
CV = (std_muestra / media) * 100
print(f"Media: {media:.2f}")
print(f"CV: {CV:.2f}%")

if CV < 15:
    print("  Variabilidad baja (proceso consistente)")
elif CV < 30:
    print("  Variabilidad moderada")
else:
    print("  Variabilidad alta (datos muy dispersos)")

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — Sobre la "salida esperada" del apunte</p>

En el PDF original, la tabla de "salida esperada" de esta celda lista varianza poblacional 651,36 / muestral 705,64 / desviación 26,56 / CV 48,08 %. Esos valores <b>no corresponden</b> a este vector de datos: al ejecutar el código se obtiene varianza poblacional <b>684,02</b>, muestral <b>741,03</b>, desviación <b>27,22</b> y CV <b>49,29 %</b> (el rango, los cuartiles, el IQR y la media sí coinciden). Como este notebook ejecuta el código, los números que ves arriba son los correctos. Conviene corregir esa tabla en el .tex.
</div>

### 6.2 SciPy: asimetría y curtosis

In [ ]:
print("\n" + "=" * 60)
print("MEDIDAS DE FORMA")
print("=" * 60)

# Asimetría
skewness = stats.skew(datos)
print(f"Asimetría: {skewness:.3f}")

if abs(skewness) < 0.5:
    print("  Aproximadamente simétrica")
elif skewness > 0:
    print("  Asimetría positiva (cola hacia la derecha)")
else:
    print("  Asimetría negativa (cola hacia la izquierda)")

# Curtosis de exceso (distribución normal = 0)
kurtosis = stats.kurtosis(datos)
print(f"\nCurtosis (exceso): {kurtosis:.3f}")

if abs(kurtosis) < 0.5:
    print("  Mesocúrtica (similar a distribución normal)")
elif kurtosis > 0:
    print("  Leptocúrtica (colas pesadas, mayor frecuencia de extremos)")
else:
    print("  Platicúrtica (colas ligeras, menor frecuencia de extremos)")

<div style="background-color:#fffceb; border-left:5px solid #f9a825; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#f57f17; font-size:1.05em;">📝&nbsp; Nota — SciPy y pandas no dan exactamente el mismo número</p>

Las fórmulas $g_1$ y $g_2$ del apunte usan el factor $1/n$, que es justo lo que calcula <code>scipy.stats.skew</code> / <code>kurtosis</code> con su opción por defecto (<code>bias=True</code>). En cambio, <b>pandas</b> (<code>.skew()</code> / <code>.kurt()</code>) aplica la corrección de sesgo para muestras, por lo que entrega valores ligeramente distintos. Para este vector: asimetría $-0{,}109$ (scipy) vs $-0{,}124$ (pandas); curtosis $-1{,}316$ (scipy) vs $-1{,}355$ (pandas). Ninguno está "mal": responden a convenciones distintas. Para muestras pequeñas la diferencia es visible; para $n$ grande, despreciable.
</div>

### 6.3 Pandas: resumen integrado

In [ ]:
df = pd.DataFrame({'valores': datos})

print("\n" + "=" * 60)
print("RESUMEN ESTADÍSTICO CON PANDAS")
print("=" * 60)

# describe() incluye los cuartiles y estadísticos principales
print(df.describe())

# Estadísticos de forma (pandas usa ddof=1 por defecto)
print(f"\nAsimetría: {df['valores'].skew():.3f}")
print(f"Curtosis:  {df['valores'].kurt():.3f}")

# Para especificar ddof explícitamente
print(f"\nDesv. std (ddof=1): {df['valores'].std(ddof=1):.4f}")
print(f"Desv. std (ddof=0): {df['valores'].std(ddof=0):.4f}")

## 7. Ejemplo integrador: análisis de portafolios

El siguiente ejemplo aplica todas las medidas estudiadas al análisis comparativo de tres portafolios de inversión.

In [ ]:
# Simular retornos mensuales (5 años = 60 meses)
np.random.seed(42)

conservadora = np.random.normal(0.5, 1.5, 60)
balanceada   = np.random.normal(0.8, 3.0, 60)

agresiva = np.random.normal(1.2, 5.0, 60)
# Simular meses con retornos excepcionales
agresiva[np.random.choice(60, 5, replace=False)] *= 2.5

df = pd.DataFrame({
    'Conservadora': conservadora,
    'Balanceada':   balanceada,
    'Agresiva':     agresiva
})

print("=" * 70)
print("ANÁLISIS COMPARATIVO DE PORTAFOLIOS")
print("=" * 70)

print("\nESTADÍSTICOS DESCRIPTIVOS:")
print(df.describe())

print("\n" + "=" * 70)
print("MEDIDAS DE FORMA:")
print("=" * 70)

for col in df.columns:
    skew = df[col].skew()
    kurt = df[col].kurt()
    print(f"\n{col}:")
    print(f"  Asimetría: {skew:.3f}", end="")
    if abs(skew) < 0.5:
        print(" (simétrica)")
    elif skew > 0:
        print(" (cola derecha — retornos positivos extremos)")
    else:
        print(" (cola izquierda — pérdidas extremas)")
    print(f"  Curtosis:  {kurt:.3f}", end="")
    if abs(kurt) < 0.5:
        print(" (mesocúrtica)")
    elif kurt > 0:
        print(" (leptocúrtica — eventos extremos frecuentes)")
    else:
        print(" (platicúrtica — retornos concentrados)")

# Sharpe Ratio: retorno ajustado por riesgo (anualizado)
print("\n" + "=" * 70)
print("SHARPE RATIO (retorno medio / desv. std × sqrt(12)):")
print("=" * 70)
sharpe = (df.mean() / df.std()) * np.sqrt(12)
print(sharpe.sort_values(ascending=False))

print("\nInterpretación: Sharpe > 1 se considera aceptable.")
print("El portafolio con mayor Sharpe ofrece mejor retorno por unidad de riesgo.")

*Comparación visual de la distribución de retornos de los tres portafolios:*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
colores = {"Conservadora": "#2e8b57", "Balanceada": "#2e6fb0", "Agresiva": UNAB_RED}
for col in df.columns:
    ax.hist(df[col], bins=20, alpha=0.5, label=col, color=colores[col])
ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_xlabel("Retorno mensual (%)"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de retornos: a mayor dispersión, mayor riesgo", color=UNAB_NAVY)
ax.legend()
plt.tight_layout(); plt.show()

## 8. Aplicaciones por industria

| Industria | Medida | Aplicación |
|---|---|---|
| Finanzas | Desv. estándar | Volatilidad de activos. Mayor $\sigma$ implica mayor riesgo. El índice VIX la utiliza como referencia. |
| Manufactura | CV | CV $< 10\%$ indica proceso bajo control. CV $> 15\%$ señala necesidad de mejora. |
| Retail | IQR | Caracterizar ventas típicas sin que eventos extraordinarios (ofertas de temporada) distorsionen el análisis. |
| Healthcare | Asimetría | Tiempos de recuperación. Asimetría positiva indica que algunos casos requieren atención prolongada fuera de lo habitual. |
| Seguros | Curtosis | Distribución de siniestros. Curtosis alta indica mayor probabilidad de eventos catastróficos que la modelada por la normal. |
| Tecnología | Desv. estándar | Latencia de servicios. Una desviación estándar superior al 50 % de la media indica inconsistencia de rendimiento. |

## 9. Buenas prácticas

<div style="background-color:#fff4e6; border-left:5px solid #e65100; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#bf360c; font-size:1.05em;">⭐&nbsp; Importante — Recomendaciones para el análisis de dispersión y forma</p>

<ol>
<li><b>Reportar múltiples medidas.</b> No limitarse a media y desviación estándar. Incluir IQR, asimetría y curtosis para una descripción completa.</li>
<li><b>Preferir IQR ante presencia de outliers.</b> Cuando los datos contienen valores atípicos, el IQR describe mejor la variabilidad del grueso de los datos.</li>
<li><b>Verificar normalidad antes de aplicar métodos que la asumen.</b> Si $|g_1| > 1$ o $g_2 > 3$, la distribución se aleja significativamente de la normal.</li>
<li><b>Contextualizar con el CV.</b> Una desviación estándar de \$10.000 puede ser trivial (si la media es \$1.000.000) o crítica (si la media es \$50.000).</li>
<li><b>Respetar las limitaciones del CV.</b> No usarlo con escalas de intervalo (temperatura Celsius) ni cuando la media es cercana a cero.</li>
<li><b>Especificar <code>ddof</code> explícitamente en Python.</b> No confiar en los valores por defecto de NumPy (<code>ddof=0</code>) vs pandas (<code>ddof=1</code>).</li>
<li><b>Visualizar antes de calcular.</b> Un histograma revela asimetría y curtosis antes de calcular los estadísticos.</li>
</ol>
</div>

### 9.1 Checklist de análisis estadístico

Al analizar un nuevo conjunto de datos, seguir este orden garantiza una descripción completa:

1. Calcular **media** y **mediana** para caracterizar la tendencia central.
2. Calcular **desviación estándar** (dispersión principal).
3. Calcular **rango** e **IQR** para tener perspectiva de amplitud y robustez.
4. Calcular **CV** si se requiere comparar con otros conjuntos de datos.
5. Evaluar la **asimetría**: ¿hay cola pronunciada hacia algún lado?
6. Evaluar la **curtosis**: ¿hay mayor frecuencia de eventos extremos que bajo normalidad?
7. Identificar **outliers** formalmente mediante la regla $1{,}5 \times IQR$.
8. **Interpretar** todos los resultados en el contexto del problema específico.

*Función de apoyo que aplica el checklist completo a cualquier vector de datos:*

In [ ]:
def resumen_completo(x, nombre="datos"):
    """Aplica el checklist de análisis a un vector numérico."""
    x = np.asarray(x, dtype=float)
    Q1, Q3 = np.percentile(x, [25, 75])
    IQR = Q3 - Q1
    s = x.std(ddof=1)
    media = x.mean()
    lim_inf, lim_sup = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = x[(x < lim_inf) | (x > lim_sup)]

    print(f"--- Resumen de '{nombre}' (n = {len(x)}) ---")
    print(f"Media: {media:.3f}   Mediana: {np.median(x):.3f}")
    print(f"Desv. estándar (s): {s:.3f}")
    print(f"Rango: {np.ptp(x):.3f}   IQR: {IQR:.3f}")
    cv = (s / media * 100) if media != 0 else float('nan')
    print(f"CV: {cv:.2f}%")
    print(f"Asimetría (g1): {stats.skew(x):.3f}")
    print(f"Curtosis  (g2): {stats.kurtosis(x):.3f}")
    print(f"Outliers (regla 1.5*IQR): {len(outliers)} -> {np.round(outliers, 2).tolist()}")
    return None

# Demostración con el vector de la sección 6
resumen_completo([23, 45, 67, 89, 34, 56, 78, 90, 12, 45, 67, 89, 23], "ejemplo sección 6")

### Cierre

<div style="background-color:#ffffff; border-left:5px solid #0d2741; padding:14px 18px; border-radius:6px; margin:6px 0;">
<p style="margin:0 0 10px 0; font-weight:700; color:#0d2741; font-size:1.05em;">💡&nbsp; Síntesis</p>

Dos conjuntos de datos con la misma media pueden ser completamente distintos en términos de comportamiento, riesgo e interpretación. Las medidas de dispersión y forma completan la descripción estadística al responder no solo <i>dónde</i> está el centro, sino <i>qué tan confiable</i> es ese centro y <i>cómo</i> se distribuyen los datos a su alrededor. En finanzas, la dispersión cuantifica el riesgo; en manufactura, la inconsistencia del proceso; en software, la variabilidad del rendimiento. Reportar únicamente la media sin contexto de variabilidad es una descripción incompleta que puede conducir a decisiones incorrectas. El análisis riguroso exige caracterizar simultáneamente <b>tendencia central, dispersión y forma</b>.
</div>

---

## Referencias bibliográficas

- Freedman, D., Pisani, R., & Purves, R. (2007). *Statistics* (4th ed.). W. W. Norton.
- Hogg, R. V., McKean, J. W., & Craig, A. T. (2019). *Introduction to mathematical statistics* (8th ed.). Pearson.
- Taleb, N. N. (2007). *The black swan: The impact of the highly improbable*. Random House.
- Tukey, J. W. (1977). *Exploratory data analysis*. Addison-Wesley.
- Wackerly, D. D., Mendenhall, W., & Scheaffer, R. L. (2008). *Mathematical statistics with applications* (7th ed.). Cengage Learning.

---
<div style="text-align:center; color:#0d2741;"><em>Estadística Computacional para la Toma de Decisiones · MCDIA · Universidad Andrés Bello · 2026</em></div>